# exp019 — Poda de slots en inferencia (checkpoint exp012 fold 0)

SPEC: `SPEC.md` en esta carpeta. Padre: exp012. **No se entrena**: se carga `best_f0.pt` de exp012 y solo cambia qué
slots ve el modelo en inferencia (el resto se enmascara con `key_padding_mask`). Bitácora en `BITACORA.md`.

## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp019_poda_slots"
PADRE = "exp012_entrada_v2"
FOLD = 0
FOLDS = [FOLD]
SEED = 42
SPLIT = "split_v1_seed42"
SELECT_ON = "pseudo"
SMOKE = True
SMOKE_N_VAL = 40
SMOKE_N_GOLD = 20

CKPT = "/workspace/users/rodrigo/results/exp012_entrada_v2/best_f0.pt"
LABELS_CSV = "/workspace/data/labels/report_labels_v1.csv"
CACHE = "v2_05mm"
CACHE_ROOT = f"/workspace/data/cache/{CACHE}"
STUDY_META_CSV = "/workspace/data/cache/meta_v2/study_meta_v2.csv"
BACKBONE_PATH = "/workspace/models/dinov2-small"

IMG_SIZE = 336
N_SLOTS = 6
N_SLICES = 16
BATCH_SIZE = 1
N_WORKERS = 8

# Costo (docs/pasos/costo_inferencia.md, rama exp013_cabeza_hallazgo): segundos v2 por serie, 1 hilo, slots 1..6
T_SLOT_V2 = [0.325, 0.191, 0.254, 0.314, 0.245, 0.181]
PRE_6SLOTS_S = 281.0      # preprocesado v2, 6 slots, ~1300 estudios, 4 vCPU ideal
PRE_SLOTS123_S = 182.0    # idem slots 1-3 (solo para comprobar la estimación)
N_TEST = 1300
MAX_AUC = 0.965           # supuesto: 1 min = 60/32400*(MAX_AUC-0.5) ~ 0.00086 AUC
COST_MAX_FRAC = 0.65
AUC_TOL = 0.01
GOLD_VETO = 0.02
NOTES = "solo inferencia: poda de slots con key_padding_mask sobre exp012 f0"

## 1. Setup

In [ ]:
PARAMS = {k: v for k, v in dict(globals()).items() if k.isupper() and isinstance(v, (int, float, str, list, bool))}
import itertools
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.experiments import git_info, guardar_metricas, seed_everything
from src.knee.metrics import mean_auc
from src.knee.splits import load_split

assert SELECT_ON == "pseudo", "Gold nunca selecciona nada (docs/06_SDD.md)"
assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe la SPEC del experimento"
assert Path(CKPT).exists(), f"No existe el checkpoint {CKPT}"
AUTHOR = current_user()
RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(SEED)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · {torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu'}")
if GIT["src_dirty"]:
    print("OJO: src/ o SPLITS/ tienen cambios sin commit; el resultado no será reproducible.")

## 2. Datos y split (código de exp012 sin cambios; solo validación del fold y gold)

In [ ]:
import cv2
from torch.utils.data import Dataset

SLOT_COLUMNS = [
    "slot1_sag_fluid_fs",
    "slot2_cor_fluid_fs",
    "slot3_ax_fluid_fs",
    "slot4_sag_pdt2",
    "slot5_cor_t1",
    "slot6_sag_t1",
]

def sample_slice_indices(total, count, train=False):
    if total <= 0:
        return np.zeros(count, dtype=int)
    positions = np.linspace(0, total - 1, count)
    if train and total > count:
        step = (total - 1) / max(count - 1, 1)
        positions += np.random.uniform(-0.35 * step, 0.35 * step, size=count)
    return np.clip(np.round(positions), 0, total - 1).astype(int)

def prepare_slices(volume, n_slices, image_size, train=False):
    indices = sample_slice_indices(len(volume), n_slices, train)
    images = np.stack([
        cv2.resize(volume[index], (image_size, image_size), interpolation=cv2.INTER_AREA)
        for index in indices
    ]).astype(np.float32)
    if train:
        gain = np.random.uniform(0.9, 1.1)
        offset = np.random.uniform(-8, 8)
        angle = np.random.uniform(-5, 5)
        scale = np.random.uniform(0.97, 1.03)
        matrix = cv2.getRotationMatrix2D((image_size / 2, image_size / 2), angle, scale)
        images = np.stack([
            cv2.warpAffine(image, matrix, (image_size, image_size), borderMode=cv2.BORDER_REFLECT_101)
            for image in images
        ])
        images = np.clip(images * gain + offset, 0, 255)
    images = images[:, None] / 255.0
    images = np.repeat(images, 3, axis=1)
    mean = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)[None, :, None, None]
    std = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)[None, :, None, None]
    return torch.from_numpy((images - mean) / std).float()

class KneeV2Dataset(Dataset):
    def __init__(self, studies, cache_root, study_meta, labels=None, n_slots=6, n_slices=16,
                 image_size=336, train=False):
        self.studies = [str(study) for study in studies]
        self.cache_root = Path(cache_root)
        meta = pd.read_csv(study_meta, dtype={"study": str}) if not isinstance(study_meta, pd.DataFrame) else study_meta.copy()
        meta["study"] = meta.study.astype(str)
        missing_columns = set(SLOT_COLUMNS[:n_slots]) - set(meta.columns)
        if missing_columns:
            raise ValueError(f"Faltan columnas de slots: {sorted(missing_columns)}")
        self.study_meta = meta.set_index("study")
        self.n_slots = n_slots
        self.n_slices = n_slices
        self.image_size = image_size
        self.train = train
        if labels is None:
            self.targets = None
        else:
            labels_df = pd.read_csv(labels) if not isinstance(labels, pd.DataFrame) else labels.copy()
            labels_df["StudyInstanceUID"] = labels_df.StudyInstanceUID.astype(str)
            self.targets = labels_df.set_index("StudyInstanceUID")

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, index):
        study = self.studies[index]
        if study not in self.study_meta.index:
            raise KeyError(f"El estudio {study} no existe en study_meta_v2.csv")
        row = self.study_meta.loc[study]
        images = torch.zeros(self.n_slots, self.n_slices, 3, self.image_size, self.image_size)
        series_meta = torch.arange(self.n_slots, dtype=torch.long)
        series_mask = torch.ones(self.n_slots, dtype=torch.bool)
        for slot_index, column in enumerate(SLOT_COLUMNS[:self.n_slots]):
            series = row[column]
            if pd.isna(series) or not str(series).strip():
                continue
            path = self.cache_root / study / f"{series}.npy"
            volume = np.load(path, mmap_mode="r")
            if volume.ndim != 3 or volume.dtype != np.uint8:
                raise ValueError(f"Cache v2 invalido en {path}: shape={volume.shape}, dtype={volume.dtype}")
            if row.get("study_side") == "L":
                volume = volume[::-1] if slot_index in (0, 3, 5) else volume[:, :, ::-1]
            images[slot_index] = prepare_slices(volume, self.n_slices, self.image_size, self.train)
            series_mask[slot_index] = False
        if series_mask.all():
            raise RuntimeError(f"El estudio {study} no tiene slots disponibles")

        target = torch.full((len(LABELS),), float("nan"))
        if self.targets is not None and study in self.targets.index:
            values = pd.to_numeric(self.targets.loc[study, LABELS], errors="coerce").to_numpy(dtype=np.float32)
            target = torch.from_numpy(values)
        return {"study": study, "images": images, "series_meta": series_meta,
                "series_mask": series_mask, "targets": target}

split = load_split(SPLIT)
pseudo = pd.read_csv(LABELS_CSV)
pseudo["StudyInstanceUID"] = pseudo.StudyInstanceUID.astype(str)
pseudo = pseudo[["StudyInstanceUID", *LABELS]]
train_csv = pd.read_csv("/workspace/data/raw/train.csv", encoding="utf-8")
gold = train_csv.dropna(subset=LABELS, how="all")[["StudyInstanceUID", *LABELS]]
gold["StudyInstanceUID"] = gold.StudyInstanceUID.astype(str)

pool = split[split.fold >= 0].merge(pseudo, on="StudyInstanceUID", how="inner")
assert len(gold) == 58 and set(gold.StudyInstanceUID) == set(split.loc[split.is_gold == 1, "StudyInstanceUID"])
assert not set(gold.StudyInstanceUID) & set(pool.StudyInstanceUID), "gold dentro del entrenamiento"
assert set(FOLDS) <= set(range(split.attrs["n_folds"]))

study_meta = pd.read_csv(STUDY_META_CSV, dtype={"study": str})
study_meta["study"] = study_meta.study.astype(str)
train_meta = study_meta[study_meta["split"] == "train"] if "split" in study_meta else study_meta
slot_present = train_meta[SLOT_COLUMNS].notna() & train_meta[SLOT_COLUMNS].astype(str).ne("")
coverage = slot_present.mean()
n_available = slot_present.sum(axis=1)
n_zero = int((n_available == 0).sum())
zero_fraction = n_zero / max(1, len(train_meta))
print("Cobertura por slot:", {column: f"{coverage[column]:.1%}" for column in SLOT_COLUMNS})
print(f"Estudios sin slots: {n_zero}/{len(train_meta)} ({zero_fraction:.2%})")
if zero_fraction >= 0.005:
    raise RuntimeError(f"{zero_fraction:.2%} de estudios no tienen slots; se exige <0.5%")
available_studies = set(train_meta.loc[n_available > 0, "study"])
pool = pool[pool.StudyInstanceUID.isin(available_studies)].reset_index(drop=True)
gold = gold[gold.StudyInstanceUID.isin(available_studies)].reset_index(drop=True)
print(f"pool {len(pool)} estudios · gold {len(gold)} · folds {pool.fold.value_counts().sort_index().to_dict()}")

dataset_args = dict(cache_root=CACHE_ROOT, study_meta=study_meta, n_slots=N_SLOTS, n_slices=N_SLICES, image_size=IMG_SIZE)
loader_args = dict(batch_size=BATCH_SIZE, num_workers=N_WORKERS, pin_memory=True)

val_ids = pool.loc[pool.fold == FOLD, "StudyInstanceUID"].reset_index(drop=True)
gold_ids = gold.StudyInstanceUID.reset_index(drop=True)
if SMOKE:
    val_ids, gold_ids = val_ids[:SMOKE_N_VAL], gold_ids[:SMOKE_N_GOLD]
val_loader = DataLoader(KneeV2Dataset(val_ids, labels=pool, train=False, **dataset_args), shuffle=False, **loader_args)
gold_loader = DataLoader(KneeV2Dataset(gold_ids, labels=gold, train=False, **dataset_args), shuffle=False, **loader_args)
print(f"evaluación: pseudo fold {FOLD} {len(val_ids)} estudios · gold {len(gold_ids)} estudios")

## 3. Modelo (clases de exp012 sin cambios) y checkpoint

In [ ]:
from contextlib import nullcontext
from torch import nn

class AttentionPool(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)

    def forward(self, values, padding_mask=None):
        query = self.query.expand(values.shape[0], -1, -1)
        pooled, _ = self.attention(query, values, values, key_padding_mask=padding_mask, need_weights=False)
        return pooled[:, 0]

class KneeV2Model(nn.Module):
    def __init__(self, backbone=None, backbone_path=None, feature_dim=None, num_labels=12,
                 attention_heads=4, gradient_checkpointing=False, n_slots=6):
        super().__init__()
        if backbone is None and backbone_path:
            from transformers import Dinov2Model
            backbone = Dinov2Model.from_pretrained(backbone_path, local_files_only=True)
            feature_dim = feature_dim or backbone.config.hidden_size
        if backbone is None or feature_dim is None:
            raise ValueError("Indica backbone + feature_dim o backbone_path")
        self.backbone = backbone
        self.feature_dim = feature_dim
        self.slot_embedding = nn.Embedding(n_slots, feature_dim)
        self.slice_pool = AttentionPool(feature_dim, attention_heads)
        self.series_pool = AttentionPool(feature_dim, attention_heads)
        self.head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, feature_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(feature_dim, num_labels),
        )
        if gradient_checkpointing and hasattr(self.backbone, "gradient_checkpointing_enable"):
            self.backbone.gradient_checkpointing_enable()

    def freeze_backbone(self, frozen=True):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = not frozen

    def autocast_context(self, device):
        if torch.device(device).type == "cuda":
            return torch.autocast("cuda", dtype=torch.bfloat16)
        return nullcontext()

    def _encode(self, images):
        output = self.backbone(pixel_values=images) if hasattr(self.backbone, "config") else self.backbone(images)
        if hasattr(output, "last_hidden_state"):
            return output.last_hidden_state[:, 0]
        if isinstance(output, (tuple, list)):
            output = output[0]
        if output.ndim == 4:
            output = output.mean(dim=(-2, -1))
        return output

    def forward(self, images, series_meta, series_mask):
        batch, n_series, n_slices, channels, height, width = images.shape
        features = self._encode(images.reshape(-1, channels, height, width))
        features = features.reshape(batch * n_series, n_slices, self.feature_dim)
        series_features = self.slice_pool(features).reshape(batch, n_series, self.feature_dim)
        series_features = series_features + self.slot_embedding(series_meta.long())
        study_features = self.series_pool(series_features, padding_mask=series_mask.bool())
        return self.head(study_features)

def build_model():
    model = KneeV2Model(backbone_path=BACKBONE_PATH, gradient_checkpointing=True, n_slots=N_SLOTS)
    return model.to(DEVICE)

model = build_model()
state = torch.load(CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(state["model"])
model.eval()
CKPT_INFO = {"path": CKPT, "exp": state.get("exp"), "fold": state.get("fold"), "epoch": state.get("epoch")}
assert CKPT_INFO["fold"] in (None, FOLD), f"El checkpoint es del fold {CKPT_INFO['fold']}, no del {FOLD}"
print("Checkpoint:", CKPT_INFO)
del state

## 4. Entrenamiento: no aplica. Búsqueda de subconjuntos de slots

Los slots no interactúan antes de `series_pool`: se calcula una vez por estudio y slot la salida de
`backbone + slice_pool` (tiempo GPU medido por serie) y cada subconjunto se evalúa solo con `slot_embedding +
series_pool + head` y su máscara. Se comprueba contra el forward completo de exp012.

In [ ]:
SLOT_NAMES = [f"s{i + 1}" for i in range(N_SLOTS)]
T_SLOT = torch.tensor(T_SLOT_V2, dtype=torch.float32)
WARMUP_BATCHES = 3

@torch.no_grad()
def extract(loader, keep_images=0):
    feats, present, targets, studies, times, kept = [], [], [], [], [], []
    for index, batch in enumerate(loader):
        images = batch["images"].to(DEVICE, non_blocking=True)
        absent = batch["series_mask"]
        out = torch.zeros(images.shape[0], N_SLOTS, model.feature_dim)
        for slot in range(N_SLOTS):
            rows = (~absent[:, slot]).nonzero().flatten()
            if len(rows) == 0:
                continue
            x = images[rows.to(DEVICE), slot]
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            with model.autocast_context(DEVICE):
                f = model._encode(x.reshape(-1, *x.shape[2:])).reshape(x.shape[0], x.shape[1], model.feature_dim)
                f = model.slice_pool(f)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            if index >= WARMUP_BATCHES:
                times.append((time.perf_counter() - start) / len(rows))
            out[rows] = f.float().cpu()
        feats.append(out)
        present.append(~absent)
        targets.append(batch["targets"])
        studies.extend(batch["study"])
        if len(kept) < keep_images:
            kept.append({k: batch[k] for k in ("images", "series_meta", "series_mask")})
    return {"feats": torch.cat(feats), "present": torch.cat(present), "y": torch.cat(targets).numpy(),
            "studies": studies, "times": times, "kept": kept}

t0 = time.time()
DATA = {"pseudo": extract(val_loader, keep_images=4), "gold": extract(gold_loader)}
EXTRACT_HOURS = (time.time() - t0) / 3600
SERIES_TIMES = DATA["pseudo"]["times"] + DATA["gold"]["times"]
T_GPU_SERIES = float(np.median(SERIES_TIMES))
print(f"extracción {EXTRACT_HOURS * 60:.1f} min · GPU por serie mediana {T_GPU_SERIES * 1000:.1f} ms "
      f"(p10 {np.percentile(SERIES_TIMES, 10) * 1000:.1f}, p90 {np.percentile(SERIES_TIMES, 90) * 1000:.1f}, n={len(SERIES_TIMES)})")
print("cobertura de slots (pseudo):", dict(zip(SLOT_NAMES, DATA["pseudo"]["present"].float().mean(0).round(decimals=3).tolist())))

def used_slots(present, subset):
    chosen = torch.zeros(N_SLOTS, dtype=torch.bool)
    chosen[list(subset)] = True
    use = present & chosen
    empty = ~use.any(1)
    if empty.any():  # regla de la SPEC: si no hay ningún slot del subconjunto, el slot presente de menor número
        first = present.float().argmax(1)
        use[empty, first[empty]] = True
    return use, empty

@torch.no_grad()
def predict_subset(data, subset, chunk=256):
    use, empty = used_slots(data["present"], subset)
    slot_ids = torch.arange(N_SLOTS, device=DEVICE)
    preds = []
    for i in range(0, len(use), chunk):
        f = data["feats"][i:i + chunk].to(DEVICE)
        m = use[i:i + chunk].to(DEVICE)
        with model.autocast_context(DEVICE):
            series = f + model.slot_embedding(slot_ids)[None]
            logits = model.head(model.series_pool(series, padding_mask=~m))
        preds.append(logits.sigmoid().float().cpu().numpy())
    return np.concatenate(preds), use, empty

# Comprobación: el atajo coincide con el forward completo de exp012 (mismo enmascarado)
check = []
for kept_index, b in enumerate(DATA["pseudo"]["kept"]):
    for subset in [tuple(range(N_SLOTS)), (0, 1, 2), (1,)]:
        present = ~b["series_mask"]
        use, _ = used_slots(present, subset)
        with torch.no_grad(), model.autocast_context(DEVICE):
            full = model(b["images"].to(DEVICE), b["series_meta"].to(DEVICE), (~use).to(DEVICE)).sigmoid().float().cpu().numpy()
        sub = {"feats": DATA["pseudo"]["feats"][kept_index:kept_index + 1], "present": present}
        fast, _, _ = predict_subset(sub, subset)
        check.append(float(np.abs(full - fast).max()))
print(f"atajo vs forward completo: máx |Δprob| = {max(check):.2e} en {len(check)} casos")
assert max(check) < 2e-2, "El atajo por features no reproduce el forward completo"
del DATA["pseudo"]["kept"]

raw_full = None
def evaluate(subset):
    global raw_full
    out = {"subset": tuple(subset), "name": "+".join(SLOT_NAMES[s] for s in subset), "n_slots": len(subset)}
    for split_name, data in DATA.items():
        preds, use, empty = predict_subset(data, subset)
        auc = mean_auc(data["y"], preds)
        out[f"{split_name}_auc"] = auc["mean"]
        out[f"{split_name}_per_label"] = auc["per_label"]
        if split_name == "pseudo":
            out["_preds"] = preds
            raw = float((use.float() * T_SLOT).sum(1).mean())
            if raw_full is None and len(subset) == N_SLOTS:
                raw_full = raw
            out["pre_raw"] = raw
            out["series_per_study"] = float(use.float().sum(1).mean())
            out["fallback_frac"] = float(empty.float().mean())
    return out

RESULTS_BY_SUBSET = {}
def get(subset):
    subset = tuple(sorted(subset))
    if subset not in RESULTS_BY_SUBSET:
        RESULTS_BY_SUBSET[subset] = evaluate(subset)
    return RESULTS_BY_SUBSET[subset]

FULL = tuple(range(N_SLOTS))
get(FULL)
# grupo 1: un slot
SINGLES = [(s,) for s in range(N_SLOTS)]
# grupo 2: eliminación hacia atrás greedy por pseudo AUC macro (empate: menor costo)
BACKWARD = [FULL]
current = FULL
while len(current) > 1:
    options = [tuple(s for s in current if s != drop) for drop in current]
    current = max(options, key=lambda sub: (get(sub)["pseudo_auc"], -get(sub)["pre_raw"]))
    BACKWARD.append(current)
# grupo 3: slots 1-3
SLOTS123 = (0, 1, 2)
# resto (incluye los ≤2 slots para el análisis por hallazgo); no participan en la elección
for size in range(1, N_SLOTS + 1):
    for subset in itertools.combinations(range(N_SLOTS), size):
        get(subset)
print("camino hacia atrás:", " → ".join(get(s)["name"] for s in BACKWARD))

## 5. Evaluación: tabla, Pareto costo vs AUC, elección por pseudo y decisión

In [ ]:
def cost(row):
    pre = PRE_6SLOTS_S * row["pre_raw"] / raw_full
    gpu = row["series_per_study"] * T_GPU_SERIES * N_TEST
    return pre, gpu

def efficiency(auc, seconds):
    return auc / (0.5 - MAX_AUC) + seconds / 32400

rows = []
for subset, r in RESULTS_BY_SUBSET.items():
    pre, gpu = cost(r)
    groups = [g for g, members in (("1slot", SINGLES), ("backward", BACKWARD), ("slots123", [SLOTS123])) if subset in members]
    rows.append({"subset": r["name"], "slots": list(subset), "n_slots": r["n_slots"], "groups": ",".join(groups),
                 "candidate": bool(groups), "pseudo_auc": r["pseudo_auc"], "gold_auc": r["gold_auc"],
                 "pre_s": pre, "model_s": gpu, "total_s": pre + gpu,
                 "series_per_study": r["series_per_study"], "fallback_frac": r["fallback_frac"]})
table = pd.DataFrame(rows)
base = table.loc[table.n_slots == N_SLOTS].iloc[0]
table["cost_frac"] = table.total_s / base.total_s
table["d_pseudo"] = table.pseudo_auc - base.pseudo_auc
table["d_gold"] = table.gold_auc - base.gold_auc
table["eff_pseudo"] = efficiency(table.pseudo_auc, table.total_s)
table["eff_gold"] = efficiency(table.gold_auc, table.total_s)
table = table.sort_values(["candidate", "eff_pseudo"], ascending=[False, True]).reset_index(drop=True)
table.drop(columns="slots").to_csv(RESULTS / "subsets.csv", index=False)

pre123 = table.loc[table.subset == "s1+s2+s3", "pre_s"].iloc[0]
print(f"comprobación costo: preprocesado slots 1-3 estimado {pre123:.0f} s vs medido {PRE_SLOTS123_S:.0f} s")
print(f"6 slots: pseudo {base.pseudo_auc:.4f} · gold {base.gold_auc:.4f} · {base.total_s:.0f} s "
      f"(pre {base.pre_s:.0f} + modelo {base.model_s:.0f})")
display(table.loc[table.candidate].drop(columns="slots").round(4))

# Elección (solo pseudo): candidatos con costo ≤ 65 % y pseudo ≥ 6 slots − 0.01 → menor score de eficiencia (pseudo)
eligible = table[table.candidate & (table.cost_frac <= COST_MAX_FRAC) & (table.d_pseudo >= -AUC_TOL)]
if len(eligible):
    chosen = eligible.sort_values("eff_pseudo").iloc[0]
    gold_ok = chosen.d_gold >= -GOLD_VETO
    DECISION = "worked" if gold_ok else "didnt"
    reason = "cumple costo y pseudo; gold " + ("dentro" if gold_ok else "fuera") + " del veto"
else:
    chosen = base
    DECISION = "didnt"
    reason = "ningún candidato cumple costo ≤ 65 % y pseudo ≥ 6 slots − 0.01"
CHOSEN = tuple(int(s) for s in chosen.slots)
print(f"Elegido: {chosen.subset} · pseudo {chosen.pseudo_auc:.4f} (Δ {chosen.d_pseudo:+.4f}) · gold {chosen.gold_auc:.4f} "
      f"(Δ {chosen.d_gold:+.4f}) · costo {chosen.total_s:.0f} s ({chosen.cost_frac:.0%})")
print("Decisión automática:", DECISION, "·", reason)

# Por hallazgo: mejor combinación de ≤2 slots según la pseudo AUC de ese hallazgo
per_finding = []
small = [s for s in RESULTS_BY_SUBSET if len(s) <= 2]
for label in LABELS:
    valid = [s for s in small if np.isfinite(RESULTS_BY_SUBSET[s]["pseudo_per_label"][label])]
    if not valid:
        continue
    best = max(valid, key=lambda s: RESULTS_BY_SUBSET[s]["pseudo_per_label"][label])
    r = RESULTS_BY_SUBSET[best]
    per_finding.append({"label": label, "best_le2": r["name"], "pseudo_auc": r["pseudo_per_label"][label],
                        "pseudo_auc_6slots": RESULTS_BY_SUBSET[FULL]["pseudo_per_label"][label],
                        "gold_auc": r["gold_per_label"][label],
                        "gold_auc_6slots": RESULTS_BY_SUBSET[FULL]["gold_per_label"][label]})
per_finding = pd.DataFrame(per_finding)
per_finding["d_pseudo"] = per_finding.pseudo_auc - per_finding.pseudo_auc_6slots
per_finding.to_csv(RESULTS / "per_finding_le2.csv", index=False)
display(per_finding.round(4))

# Pareto costo vs AUC
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
for ax, metric in zip(axes, ["pseudo_auc", "gold_auc"]):
    other = table[~table.candidate]
    ax.scatter(other.total_s, other[metric], c="0.75", s=18, label="otros subconjuntos")
    for group, color in (("1slot", "tab:blue"), ("backward", "tab:orange"), ("slots123", "tab:green")):
        g = table[table.groups.str.contains(group)]
        ax.scatter(g.total_s, g[metric], c=color, s=40, label=group)
    path = table.set_index("subset").loc[[get(s)["name"] for s in BACKWARD]]
    ax.plot(path.total_s, path[metric], c="tab:orange", lw=1)
    front = table.sort_values("total_s")
    front = front[front.pseudo_auc.cummax() == front.pseudo_auc]  # frente Pareto definido por pseudo
    ax.step(front.total_s, front[metric], where="post", c="k", lw=0.8, ls="--", label="frente (pseudo)")
    ax.axvline(COST_MAX_FRAC * base.total_s, c="r", lw=0.8, ls=":")
    ax.axhline(base[metric] - (AUC_TOL if metric == "pseudo_auc" else GOLD_VETO), c="r", lw=0.8, ls=":")
    ax.scatter([chosen.total_s], [chosen[metric]], marker="*", s=200, c="red", label=f"elegido {chosen.subset}")
    for _, r in table[table.candidate].iterrows():
        ax.annotate(r.subset, (r.total_s, r[metric]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel(f"costo estimado Kaggle (s, {N_TEST} estudios)")
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_title(f"{EXP} · {metric.split('_')[0]}{' (SMOKE)' if SMOKE else ''}")
axes[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(RESULTS / "pareto_costo_auc.png", dpi=130)
plt.show()

## 6. Guardar `results/<EXP>/metrics.json`

In [ ]:
chosen_r = RESULTS_BY_SUBSET[CHOSEN]
full_r = RESULTS_BY_SUBSET[FULL]
per_fold = {str(FOLD): {
    "pseudo_auc": chosen_r["pseudo_auc"], "gold_auc": chosen_r["gold_auc"], "best_epoch": CKPT_INFO["epoch"],
    "pseudo_per_label": chosen_r["pseudo_per_label"], "gold_per_label": chosen_r["gold_per_label"],
    "gpu_hours": EXTRACT_HOURS, "ckpt_path": CKPT, "subset": chosen.subset,
}}
metrics = {
    "schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE,
    "split": SPLIT, "split_sha256": split.attrs["sha256"], "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else "screen",
    "select_on": SELECT_ON, "params": PARAMS, "per_fold": per_fold,
    "gold_auc": chosen_r["gold_auc"], "gold_per_label": chosen_r["gold_per_label"], "oof_pseudo_auc": None,
    "delta": {"pseudo_f0": float(chosen.d_pseudo), "gold": float(chosen.d_gold)},
    "decision_auto": DECISION,
    "gpu_hours_total": EXTRACT_HOURS,
    "checkpoint": CKPT_INFO,
    "chosen_subset": chosen.subset,
    "baseline_6slots": {"pseudo_auc": full_r["pseudo_auc"], "gold_auc": full_r["gold_auc"],
                        "pseudo_per_label": full_r["pseudo_per_label"], "gold_per_label": full_r["gold_per_label"],
                        "total_s": float(base.total_s), "pre_s": float(base.pre_s), "model_s": float(base.model_s)},
    "backward_path": [get(s)["name"] for s in BACKWARD],
    "cost_model": {"t_slot_v2_s": T_SLOT_V2, "pre_6slots_s": PRE_6SLOTS_S, "pre_slots123_measured_s": PRE_SLOTS123_S,
                   "pre_slots123_estimated_s": float(pre123), "t_gpu_series_s": T_GPU_SERIES,
                   "gpu": torch.cuda.get_device_name() if DEVICE.type == "cuda" else "cpu",
                   "n_test": N_TEST, "max_auc": MAX_AUC, "n_eval_pseudo": len(DATA["pseudo"]["studies"]),
                   "n_eval_gold": len(DATA["gold"]["studies"])},
    "subsets": table.drop(columns="slots").to_dict(orient="records"),
    "per_finding_le2": per_finding.to_dict(orient="records"),
}
print("Guardado:", guardar_metricas(RESULTS, metrics))

## 7. Verdict

Copia los números de la celda 5 a la sección 3 de `SPEC.md` (6 slots vs subconjunto elegido) y actualiza la fila de
exp019 en `docs/EXPERIMENTOS.md` (`worked`, `didnt` o `killed`) con una línea de qué aprendimos. Recuerda: el
subconjunto se elige por pseudo; gold solo veta. Después: commit y PR a `main`.